In [3]:
from pathlib import Path
list(Path("../data/raw").glob("*.docx"))

[PosixPath('../data/raw/Flh Bln Chronik 1989 - 2017 bis 13.dez.docx')]

In [4]:
from pathlib import Path
import re
import pandas as pd
from docx import Document

In [5]:
raw_docs = list(Path("../data/raw").glob("*.docx"))
raw_docs

[PosixPath('../data/raw/Flh Bln Chronik 1989 - 2017 bis 13.dez.docx')]

In [9]:
from pathlib import Path

DOCS = list(Path("../data/raw").glob("*.docx"))

if len(DOCS) == 0:
    raise FileNotFoundError("Keine DOCX-Datei in data/raw gefunden.")

if len(DOCS) > 1:
    print("Mehrere DOCX-Dateien gefunden, nehme die erste:", DOCS[0])

DOC_PATH = DOCS[0]
DOC_PATH

PosixPath('../data/raw/Flh Bln Chronik 1989 - 2017 bis 13.dez.docx')

In [10]:
YEAR_RE = re.compile(r"^(19|20)\d{2}$")

doc = Document(DOC_PATH)

rows = []
year_bucket = None
year_section_index = 0
global_index = 0

for p in doc.paragraphs:
    text = (p.text or "").strip()
    if not text:
        continue

    # Jahr-Überschrift
    if YEAR_RE.match(text):
        year_bucket = int(text)
        year_section_index = 0
        global_index += 1
        rows.append({
            "id": global_index,
            "doc_anchor": f"p{global_index}",
            "year_bucket": year_bucket,
            "year_section_index": 0,
            "text_span": text,
            "is_year_heading": True,
        })
        continue

    year_section_index += 1
    global_index += 1
    rows.append({
        "id": global_index,
        "doc_anchor": f"p{global_index}",
        "year_bucket": year_bucket,
        "year_section_index": year_section_index,
        "text_span": text,
        "is_year_heading": False,
    })

df = pd.DataFrame(rows)
df.head(10)

,id,doc_anchor,year_bucket,year_section_index,text_span,is_year_heading
0,1,p1,NaN,1,Flughafen Berlin-Brandenburg,False
1,2,p2,NaN,2,Chronik der Entwicklung 1989 bis 2017,False
2,3,p3,NaN,3,Ausgewertet und zusammengestellt von André Geicke,False
3,4,p4,1989.0,0,1989,True
4,5,p5,1989.0,1,29. Januar Wahl zum Berliner Abgeordnetenhaus....,False
5,6,p6,1989.0,2,Berlin brauche noch vor dem Jahr 2000 einen ne...,False
6,7,p7,1989.0,3,21. April Friedrich Zimmermann ist neuer Bunde...,False
7,8,p8,1989.0,4,Nach Information des Berliner Verkehrssenators...,False
8,9,p9,1989.0,5,„Daß die Weltkriegssieger auf ihre alten Besat...,False
9,10,p10,1990.0,0,1990,True
